In [32]:
import os
import numpy as np
import pandas as pd
from scipy.sparse import load_npz, save_npz
from sklearn.preprocessing import normalize

In [33]:
tfidf_matrix = load_npz("../data/output/TFIDF.npz")

print("TFIDF shape:", tfidf_matrix.shape)

TFIDF shape: (31102, 12878)


In [34]:
term_df = pd.read_csv("../data/output/tfidf_term_index.csv")

print(term_df.head())
print("Full vocab shape:", term_df.shape)

  Unnamed: 0  0
0        and  0
1  beginning  1
2    created  2
3      earth  3
4        god  4
Full vocab shape: (12878, 2)


In [35]:
full_terms = term_df.iloc[:, 0].tolist()

print("Number of full vocabulary terms:", len(full_terms))
print("Sample full terms:", full_terms[:5])

Number of full vocabulary terms: 12878
Sample full terms: ['and', 'beginning', 'created', 'earth', 'god']


In [36]:
k = 1000

# Sum TFIDF across documents
term_scores = np.array(tfidf_matrix.sum(axis=0)).flatten()

# Get indices of top-k terms
top_term_indices = np.argsort(term_scores)[-k:]

# Reduce matrix
reduced_tfidf = tfidf_matrix[:, top_term_indices]

print("Reduced TFIDF shape:", reduced_tfidf.shape)

Reduced TFIDF shape: (31102, 1000)


In [37]:
tfidf_l2 = normalize(reduced_tfidf, norm="l2", axis=1)

print("L2 Normalized shape:", tfidf_l2.shape)

L2 Normalized shape: (31102, 1000)


In [38]:
row_norms = np.linalg.norm(tfidf_l2.toarray(), axis=1)
print("Min row norm:", row_norms.min())
print("Max row norm:", row_norms.max())

Min row norm: 0.0
Max row norm: 1.0000000000000002


In [41]:
delimiter = "|"

print("Delimiter:", delimiter)
print("Number of features (i.e. significant words):", k)

principle_text = f"""
Principle of significant word selection:

Top-k terms ranked by total TFIDF weight across all documents,
retaining the top {k} highest-scoring terms.

Significant words were selected by ranking terms according to
their total TFIDF weight across all documents and retaining
the top k terms, prioritizing features that are both informative
and substantively present in the corpus.
"""

print(principle_text)

Delimiter: |
Number of features (i.e. significant words): 1000

Principle of significant word selection:

Top-k terms ranked by total TFIDF weight across all documents,
retaining the top 1000 highest-scoring terms.

Significant words were selected by ranking terms according to
their total TFIDF weight across all documents and retaining
the top k terms, prioritizing features that are both informative
and substantively present in the corpus.



In [39]:
os.makedirs("../data/output", exist_ok=True)

save_npz("../data/output/TFIDF_L2_REDUCED.npz", tfidf_l2)

print("Reduced L2 TFIDF saved.")

Reduced L2 TFIDF saved.


In [40]:
reduced_terms = [full_terms[i] for i in top_term_indices]

print("Sample reduced terms:", reduced_terms[:5])
print("Reduced term count:", len(reduced_terms))

pd.Series(reduced_terms).to_csv(
    "../data/output/tfidf_term_index_reduced.csv",
    index=False
)

print("Reduced term index saved correctly.")


Sample reduced terms: ['blemish', 'thousands', 'howbeit', 'fail', 'parts']
Reduced term count: 1000
Reduced term index saved correctly.
